# DOE tutorial — team analysis notebook

Load your team's completed design + responses, fit the model your archetype supports, and predict your best formulation.

**How to use in Colab:**
1. Upload your team template `.xlsx` (`team-template-OFAT.xlsx`, `-FRACTIONAL.xlsx`, or `-CCD.xlsx`) to the Colab session.
2. Set `TEAM_ARCHETYPE` below to your team's archetype string.
3. Run all cells top-to-bottom.

**Locally:** open this notebook in Jupyter/VS Code, put your filled-in `.xlsx` next to it, adjust `TEAM_ARCHETYPE`, run all cells.

In [ ]:
TEAM_ARCHETYPE = 'CCD'   # one of: 'OFAT', 'FRAC', 'CCD'
XLSX_PATH      = 'team-template-CCD.xlsx'   # path to YOUR filled-in template

assert TEAM_ARCHETYPE in ('OFAT', 'FRAC', 'CCD'), TEAM_ARCHETYPE

### Factor assignment

Record which real factor you assigned to each coded slot. This gets used at the end of the notebook when you convert your predicted-best coded point into natural units for submission.

In [ ]:
# Fill in your factor-to-slot mapping (FRAC and CCD teams only).
# For each slot A-E, put the real factor you assigned to it.
# Valid factors: 'latex_pct', 'filler_phr', 'crosslinker_phr',
#                'plasticizer_phr', 'cure_temp_c'
#
# FRAC teams: one slot (typically E) stays at midpoint; assign it anyway.
# CCD teams: one slot (typically E) stays at midpoint; assign it anyway.
# OFAT teams: you don't use slots — leave this as-is; it's ignored for OFAT.
FACTOR_ASSIGNMENT = {
    'A': None,   # <- fill in (FRAC/CCD only)
    'B': None,   # <- fill in (FRAC/CCD only)
    'C': None,   # <- fill in (FRAC/CCD only)
    'D': None,   # <- fill in (FRAC/CCD only)
    'E': None,   # <- fill in (FRAC/CCD only)
}

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print('statsmodels not available — using numpy least-squares only')

## 1. Load your data

This reads the `runs` sheet of your team template. The layout differs by archetype:

- **OFAT**: header row 1; columns are `run`, `swept_factor`, five natural-unit factor columns, three response columns.
- **FRAC / CCD**: header row 10 (factor-assignment block sits above); columns include `coded_A..E`, five naturals, and three responses.

Rows without recorded responses are dropped.

In [ ]:
if TEAM_ARCHETYPE == 'OFAT':
    # OFAT runs sheet: header at row 1, natural-unit columns only.
    df = pd.read_excel(XLSX_PATH, sheet_name='runs', header=0)
    df = df.dropna(subset=['tensile_mpa'])
else:
    # FRAC/CCD runs sheet: factor-assignment block above; header at row 10.
    df = pd.read_excel(XLSX_PATH, sheet_name='runs', header=9)
    df = df.dropna(subset=['coded_A', 'coded_B', 'coded_C', 'coded_D', 'coded_E'])
print(f'{len(df)} runs loaded.')
df.head()

## 2. Fit the model your design supports

- **OFAT**: for each factor you swept, find the natural-unit value that gave the best tensile. Combine those per-factor bests into a predicted joint optimum. No interactions, no curvature.
- **FRAC**: linear + 2-way interactions from the 2⁴ corners on slots A, B, C, D. Lack-of-fit test if center points were included.
- **CCD**: full quadratic model on slots A, B, C, D — main effects, all six 2-way interactions, four pure-quadratic terms. That is 15 parameters against 15 observations (saturated fit); trust coefficient magnitudes but not p-values.

In [ ]:
def fit_and_report(df, response, archetype):
    rows = df.dropna(subset=[response]).copy()

    if archetype == 'OFAT':
        # For each factor the team actually swept, group by the natural-unit
        # value and report the best response value at each level.
        results = {}
        swept = rows['swept_factor'].dropna().astype(str).str.strip().unique()
        real_factors = ['latex_pct', 'filler_phr', 'crosslinker_phr',
                        'plasticizer_phr', 'cure_temp_c']
        for f in real_factors:
            f_rows = rows[rows['swept_factor'].astype(str).str.strip() == f]
            if len(f_rows) == 0:
                continue  # this factor was not swept
            best_idx = f_rows[response].idxmax()
            best_val = f_rows.loc[best_idx, f]
            best_resp = f_rows.loc[best_idx, response]
            results[f] = dict(best_natural_value=best_val,
                               best_response=best_resp,
                               n_levels_tried=len(f_rows))
        return dict(kind='ofat', per_factor_best=results, swept_factors=list(swept))

    if archetype == 'FRAC':
        # Linear + 2-way interactions on the 2^4 corners of slots A, B, C, D.
        corners = rows[rows[[f'coded_{x}' for x in 'ABCD']].abs().eq(1).all(axis=1)]
        centers = rows[rows[[f'coded_{x}' for x in 'ABCDE']].eq(0).all(axis=1)]
        X_terms = ['A','B','C','D','AB','AC','AD','BC','BD','CD']
        def design_row(r):
            base = {t: r[f'coded_{t}'] for t in 'ABCD'}
            for a,b in itertools.combinations('ABCD', 2):
                base[a+b] = r[f'coded_{a}'] * r[f'coded_{b}']
            return base
        Xc = pd.DataFrame([design_row(r) for _, r in corners.iterrows()])[X_terms]
        Xc.insert(0, 'intercept', 1)
        yc = corners[response].values
        beta, *_ = np.linalg.lstsq(Xc.values, yc, rcond=None)
        mc = corners[response].mean() if len(corners) else np.nan
        m0 = centers[response].mean() if len(centers) else np.nan
        sd0 = centers[response].std() if len(centers) > 1 else np.nan
        curvature = (m0 - mc) if len(centers) else None
        detected = (curvature is not None and abs(curvature) > 2 * (sd0 or 1e-9))
        return dict(kind='frac',
                    coefs=dict(zip(['intercept'] + X_terms, beta)),
                    lof=dict(mean_corner=mc, mean_center=m0, sd_center=sd0,
                             curvature=curvature, detected=detected))

    if archetype == 'CCD':
        # Full quadratic on 4 varied slots A, B, C, D. Slot E always 0.
        terms = [
            ('A', lambda r: r['coded_A']),
            ('B', lambda r: r['coded_B']),
            ('C', lambda r: r['coded_C']),
            ('D', lambda r: r['coded_D']),
            ('AB', lambda r: r['coded_A']*r['coded_B']),
            ('AC', lambda r: r['coded_A']*r['coded_C']),
            ('AD', lambda r: r['coded_A']*r['coded_D']),
            ('BC', lambda r: r['coded_B']*r['coded_C']),
            ('BD', lambda r: r['coded_B']*r['coded_D']),
            ('CD', lambda r: r['coded_C']*r['coded_D']),
            ('AA', lambda r: r['coded_A']**2),
            ('BB', lambda r: r['coded_B']**2),
            ('CC', lambda r: r['coded_C']**2),
            ('DD', lambda r: r['coded_D']**2),
        ]
        X = np.array([[fn(r) for _, fn in terms] for _, r in rows.iterrows()])
        X = np.column_stack([np.ones(len(X)), X])
        y = rows[response].values
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        return dict(kind='ccd', coefs=dict(zip(['intercept'] + [t for t,_ in terms], beta)))

tensile_fit = fit_and_report(df, 'tensile_mpa', TEAM_ARCHETYPE)
print('Tensile fit:'); [print(f'  {k}: {v}') for k, v in tensile_fit.items()]

try:
    elong_fit = fit_and_report(df, 'elongation_pct', TEAM_ARCHETYPE)
    print('\nElongation fit:'); [print(f'  {k}: {v}') for k, v in elong_fit.items()]
except Exception as e:
    print('elongation fit skipped:', e)

## 3. Predict your best formulation

Scan a grid across the coded design space and pick the point predicted to maximize your objective (tensile).

In [ ]:
def predict_ccd(coefs, A, B, C, D):
    return (coefs.get('intercept', 0)
            + coefs.get('A',0)*A + coefs.get('B',0)*B + coefs.get('C',0)*C + coefs.get('D',0)*D
            + coefs.get('AB',0)*A*B + coefs.get('AC',0)*A*C + coefs.get('AD',0)*A*D
            + coefs.get('BC',0)*B*C + coefs.get('BD',0)*B*D + coefs.get('CD',0)*C*D
            + coefs.get('AA',0)*A*A + coefs.get('BB',0)*B*B
            + coefs.get('CC',0)*C*C + coefs.get('DD',0)*D*D)

def predict_frac(coefs, A, B, C, D):
    y = coefs.get('intercept', 0) + coefs.get('A',0)*A + coefs.get('B',0)*B + coefs.get('C',0)*C + coefs.get('D',0)*D
    for x, z in [('A','B'),('A','C'),('A','D'),('B','C'),('B','D'),('C','D')]:
        y += coefs.get(x+z, 0) * locals()[x] * locals()[z]
    return y

if TEAM_ARCHETYPE == 'OFAT':
    per_factor = tensile_fit['per_factor_best']
    print('Your OFAT-predicted best-per-factor values (natural units):')
    for f, info in per_factor.items():
        print(f'  {f}: best value = {info["best_natural_value"]:.2f} '
              f'(observed tensile = {info["best_response"]:.2f})')
    print('\nJoint prediction (combine per-factor bests):')
    joint = {f: info['best_natural_value'] for f, info in per_factor.items()}
    for f, v in joint.items():
        print(f'  {f}: {v:.2f}')
elif TEAM_ARCHETYPE == 'FRAC':
    coefs = tensile_fit['coefs']
    grid = itertools.product([-1, 0, 1], repeat=4)
    best = max(((a,b,c,d) for a,b,c,d in grid), key=lambda p: predict_frac(coefs, *p))
    print('Your FRAC-predicted best tensile point (coded A,B,C,D):', best)
    lof = tensile_fit['lof']
    if lof and lof['detected']:
        print('CURVATURE DETECTED on tensile — a linear model may be inadequate.')
elif TEAM_ARCHETYPE == 'CCD':
    coefs = tensile_fit['coefs']
    grid = list(itertools.product(np.linspace(-1, 1, 11), repeat=4))
    best_t = max(grid, key=lambda p: predict_ccd(coefs, *p))
    print(f'CCD-predicted best tensile point (coded A,B,C,D): '
          f'{tuple(round(x,3) for x in best_t)}')

## 4. What to hand in to the facilitator

Convert your predicted best coded point to natural units. This is your team's predicted-best formulation — hand it to the facilitator.

In [ ]:
RANGES = {
    'latex_pct':        (5.0, 20.0),
    'filler_phr':       (0.0, 40.0),
    'crosslinker_phr':  (0.5,  4.0),
    'plasticizer_phr':  (0.0, 25.0),
    'cure_temp_c':      (100.0, 160.0),
}

def coded_to_natural(factor, x_c):
    lo, hi = RANGES[factor]
    return 0.5*(lo+hi) + x_c * 0.5*(hi-lo)

REAL_FACTORS = ['latex_pct', 'filler_phr', 'crosslinker_phr', 'plasticizer_phr', 'cure_temp_c']

if TEAM_ARCHETYPE == 'OFAT':
    # OFAT teams already work in natural units — just print the joint prediction.
    per_factor = tensile_fit['per_factor_best']
    print('Your submission for the facilitator (natural units):')
    for f in REAL_FACTORS:
        if f in per_factor:
            print(f'  {f}: {per_factor[f]["best_natural_value"]:.2f}')
        else:
            lo, hi = RANGES[f]
            mid = 0.5*(lo+hi)
            print(f'  {f}: {mid:.2f}  (not swept — use midpoint)')
else:
    # FRAC/CCD teams — need FACTOR_ASSIGNMENT to translate coded slots back to real factors.
    if any(v is None for v in FACTOR_ASSIGNMENT.values()):
        print('WARNING: FACTOR_ASSIGNMENT still has None entries. Fill it in at the top of the notebook,')
        print('then re-run this cell.')
    else:
        # Replace `example` with your predicted-best coded point from cell above.
        # For CCD: (coded_A, coded_B, coded_C, coded_D), slot E always 0.
        example = dict(A=0.0, B=-0.5, C=0.4, D=0.4, E=0.0)
        submission = {}
        for slot, coded_val in example.items():
            real = FACTOR_ASSIGNMENT[slot]
            submission[real] = coded_to_natural(real, coded_val)
        print('Your natural-unit submission:')
        for f in REAL_FACTORS:
            print(f'  {f}: {submission[f]:.2f}')